Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

FIND THE ABSOLUTE dataset.csv FILE PATH and testing prints

In [2]:
BASE_DIR = Path().resolve().parent
file_path = BASE_DIR / "data" / "dataset.csv"

In [16]:
print(f"Looking for dataset at: {file_path}")
print(f"File exists: {file_path.exists()}")

Looking for dataset at: /Users/ilkerbaran/PycharmProjects/interview_intel/ml/data/dataset.csv
File exists: True


Read the dataset.csv file

In [4]:
df = pd.read_csv(file_path)

Basic text normalization and testing prints

In [5]:
df["text"] = df["text"].str.strip().str.replace(r"\s+", " ", regex=True)

In [6]:
print(f"Rows: {len(df)}")
print(f"Nulls after cleanup: {df['text'].isnull().sum()}")
print(f"Sample cleaned text:\n{df['text'][0]}")

Rows: 866
Nulls after cleanup: 0
Sample cleaned text:
hi luis, we've made our decision and unfortunately won't be proceeding. thank you for the conversations. best, lee


X  and y assignments --> (Features) and (Labels) and testing print

In [7]:
X = df["text"]

y_category = df["category"]
y_urgency = df["urgency"]
y_job_field = df["job_field"]

print(f"Features:  Email text only ")
print(f"Labels: category | urgency | job_field")

Features:  Email text only 
Labels: category | urgency | job_field


train_test_split -> %80 Train %20 Test -> equal split for all labels

(Design): I evaluated each prediction task independently using its own stratified train-test split. For category, urgency, and job field, I stratified on the corresponding target label so that class distributions remained balanced in both training and test sets. This gave me more reliable evaluation for each classifier, especially because combined stratification across all three targets was too sparse for the dataset.

In [8]:
# category split
X_train_cat, X_test_cat, y_cat_train, y_cat_test = train_test_split(
    X, y_category, test_size=0.2, random_state=42, stratify=y_category
)

# urgency split
X_train_urg, X_test_urg, y_urg_train, y_urg_test = train_test_split(
    X, y_urgency, test_size=0.2, random_state=42, stratify=y_urgency
)

# job_field split
X_train_job_field, X_test_job_field, y_job_field_train, y_job_field_test = train_test_split(
    X, y_job_field, test_size=0.2, random_state=42, stratify=y_job_field
)

In [9]:
print(f"Category train/test: {len(X_train_cat)} / {len(X_test_cat)}")
print(f"Urgency train/test: {len(X_train_urg)} / {len(X_test_urg)}")
print(f"Job field train/test: {len(X_train_job_field)} / {len(X_test_job_field)}")

Category train/test: 692 / 174
Urgency train/test: 692 / 174
Job field train/test: 692 / 174


Helper func to train and compare models

In [10]:
def evaluate_models(X_train, X_test, y_train, y_test, target_label):
    """
    Train three classifiers on the same split.
    Print classification report for each.
    Return the best pipeline by accuracy.
    """
    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "MultinomialNB": MultinomialNB(),
        "LinearSVC": LinearSVC(max_iter=2000, class_weight="balanced")
    }

    best_model, best_accuracy, best_pipeline = None, 0, None

    print(f"\n{'=' * 60}")
    print(f"Target: {target_label.upper()}")
    print(f"\n{'=' * 60}")

    for name, clf in models.items():
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=5000,
                min_df=1,
                sublinear_tf=True
            )),
            ("clf", clf)
        ])

        pipeline.fit(X_train, y_train)
        PRED     = pipeline.predict(X_test)
        accuracy = accuracy_score(y_test, PRED)

        print(f"\n➡️{name} Accuracy is: {accuracy:.3f}")
        print(classification_report(y_test, PRED))

        if accuracy > best_accuracy:
            best_model, best_accuracy, best_pipeline = name, accuracy, pipeline

    print(f"\n💡💡 Best For: {target_label}: {best_model} ({best_accuracy:.3f})")
    return best_model, best_pipeline

Run it for all three labels

In [11]:
best_category_name, best_category_model = evaluate_models(
    X_train_cat, X_test_cat, y_cat_train, y_cat_test, "category"
)

best_urgency_name, best_urgency_model = evaluate_models(
    X_train_urg, X_test_urg, y_urg_train, y_urg_test, "urgency"
)

best_job_field_name, best_job_field_model = evaluate_models(
    X_train_job_field, X_test_job_field, y_job_field_train, y_job_field_test, "job_field"
)


Target: CATEGORY


➡️LogisticRegression Accuracy is: 1.000
                      precision    recall  f1-score   support

           follow_up       1.00      1.00      1.00        30
interview_invitation       1.00      1.00      1.00        29
               offer       1.00      1.00      1.00        27
  recruiter_outreach       1.00      1.00      1.00        29
           rejection       1.00      1.00      1.00        30
          scheduling       1.00      1.00      1.00        29

            accuracy                           1.00       174
           macro avg       1.00      1.00      1.00       174
        weighted avg       1.00      1.00      1.00       174


➡️MultinomialNB Accuracy is: 1.000
                      precision    recall  f1-score   support

           follow_up       1.00      1.00      1.00        30
interview_invitation       1.00      1.00      1.00        29
               offer       1.00      1.00      1.00        27
  recruiter_outreach       1.00 

Test with real email examples

In [12]:
test_cases = [
    {
        "email": "Hi Sarah, we'd love to speak with you about the Software Engineer opening on our platform team. Are you available for a Zoom conversation Tuesday at 2 PM?",
        "expected_category": "interview_invitation",
        "expected_urgency": "high",
        "expected_job_field": "software_engineering",
        "fuzzy": False
    },
    {
        "email": "Thank you again for your time. After reviewing everything, we've decided to continue with another applicant whose background more closely matches what we need right now.",
        "expected_category": "rejection",
        "expected_urgency": "low",
        "expected_job_field": "general",
        "fuzzy": False
    },
    {
        "email": "I found your background while searching for candidates and thought you might be a strong fit for our machine learning group. Would you be interested in a short introductory chat sometime this week?",
        "expected_category": "recruiter_outreach",
        "expected_urgency": "low",
        "expected_job_field": "data_science",
        "fuzzy": False
    },
    {
        "email": "Something changed on our side, so we need to move your conversation with the hiring manager. Could you do Wednesday at 3 instead of Monday?",
        "expected_category": "scheduling",
        "expected_urgency": "medium",
        "expected_job_field": "general",
        "fuzzy": True
    },
    {
        "email": "We're excited to move ahead with you for the Product Manager opportunity at Apple. Your compensation package is attached, and we'd appreciate your response by Friday.",
        "expected_category": "offer",
        "expected_urgency": "high",
        "expected_job_field": "product_management",
        "fuzzy": False
    },
    {
        "email": "Hello, I wanted to circle back regarding my application for the UX Designer role I submitted last week. I'd be glad to provide anything else you need.",
        "expected_category": "follow_up",
        "expected_urgency": "low",
        "expected_job_field": "design",
        "fuzzy": False
    },

    # ── trickier cases ──
    {
        "email": "Quick note — we were impressed by your background and would like to discuss next steps. Do you have time tomorrow afternoon?",
        "expected_category": "interview_invitation",
        "expected_urgency": "high",
        "expected_job_field": "general",
        "fuzzy": True  # could be recruiter_outreach — no clear field signal
    },
    {
        "email": "Unfortunately, we won't be continuing with your application at this stage, although we really appreciated meeting you and learning more about your experience.",
        "expected_category": "rejection",
        "expected_urgency": "low",
        "expected_job_field": "general",
        "fuzzy": True
    },
    {
        "email": "Congratulations. We'd be happy to bring you onto our finance team. I've attached the formal documents for your review.",
        "expected_category": "offer",
        "expected_urgency": "high",
        "expected_job_field": "finance",
        "fuzzy": False
    },
    {
        "email": "I'm reaching out because your experience stood out to me, and I think you could be a fit for an analytics opening we're filling. Let me know if you'd like to hear more.",
        "expected_category": "recruiter_outreach",
        "expected_urgency": "low",
        "expected_job_field": "data_science",
        "fuzzy": True
    },
    {
        "email": "Can we push our call back by a day? I have a conflict on Thursday morning now.",
        "expected_category": "scheduling",
        "expected_urgency": "medium",
        "expected_job_field": "general",
        "fuzzy": True  # very short, minimal signal for job field
    },
    {
        "email": "Just checking in to see whether there have been any updates on my application. I'm still very interested in the opportunity.",
        "expected_category": "follow_up",
        "expected_urgency": "low",
        "expected_job_field": "general",
        "fuzzy": True
    },

    # ── harder edge cases ──
    {
        "email": "We enjoyed speaking with you and want to keep things moving. Please choose one of the available time slots in the scheduling link below.",
        "expected_category": "scheduling",
        "expected_urgency": "medium",
        "expected_job_field": "general",
        "fuzzy": True  # could be interview_invitation
    },
    {
        "email": "Your profile came up during our search for backend talent. The role focuses on APIs, distributed systems, and Python services. Open to learning more?",
        "expected_category": "recruiter_outreach",
        "expected_urgency": "low",
        "expected_job_field": "software_engineering",
        "fuzzy": True
    },
    {
        "email": "We are not able to extend an offer at this time, but we will keep your information on file for future openings.",
        "expected_category": "rejection",
        "expected_urgency": "low",
        "expected_job_field": "general",
        "fuzzy": True
    },
    {
        "email": "I wanted to follow up after our recent conversation and ask whether there is anything else you need from me before a decision is made.",
        "expected_category": "follow_up",
        "expected_urgency": "low",
        "expected_job_field": "general",
        "fuzzy": False
    },
]

Test pass/fail comparison

In [13]:
# Evaluation
print(f"\n{'='*60}")
print(" MANUAL TEST RESULT ")
print(f"\n{'='*60}")

passed = failed = fuzzy = 0

for i, case in enumerate(test_cases, start=1):
    email = case["email"]
    is_fuzzy = case.get("fuzzy", False)

    # ---- Expected ----
    expected = {
        "category": case["expected_category"],
        "urgency": case["expected_urgency"],
        "job_field": case["expected_job_field"]
    }

    # ---- Predicted ----
    predicted = {
        "category": best_category_model.predict([email])[0],
        "urgency": best_urgency_model.predict([email])[0],
        "job_field": best_job_field_model.predict([email])[0]
    }

    # ---- Comparison ----
    cat_ok = predicted["category"] == expected["category"]
    urg_ok = predicted["urgency"] == expected["urgency"]
    job_field_ok = predicted["job_field"] == expected["job_field"]

    matches = sum([cat_ok, urg_ok, job_field_ok])
    all_ok = matches == 3

    if all_ok:
        status = "✅PASS"
        passed += 1
    elif is_fuzzy and matches >= 2:
        status = "❕FUZZY"
        fuzzy += 1
    else:
        status = "❗️Failed"
        failed += 1

    print(f"Test {i:02d}: {status} {'(ambiguous — model may reasonably disagree)' if is_fuzzy else ''}")
    print(f"  Email     : {email[:60]}...")
    print(f"  category  -> Got: {predicted['category']:<25}"
                        f"Expected: {expected['category']:<25} "
                        f"{'✅' if cat_ok else '⚠️'}")
    print(f"  urgency   -> Got: {predicted['urgency']:<25}"
                        f"Expected: {expected['urgency']:<25} "
                        f"{'✅' if urg_ok else '⚠️'}")
    print(f"  job_field -> Got: {predicted['job_field']:<25}"
                        f"Expected: {expected['job_field']:<25} "
                        f"{'✅' if job_field_ok else '⚠️'}")
    print(f"  matched   : {matches}/3\n")

total = len(test_cases)

print(f"\n{'='*60}")
print(f" ✅Passed  : {passed} ({passed/total:.1%})")
print(f" ❕Fuzzy   : {fuzzy} ({fuzzy/total:.1%})")
print(f" ⚠️Failed  : {failed} ({failed/total:.1%})")
print(f" Total     : {total}")
print(f"\n{'='*60}")


 MANUAL TEST RESULT 

Test 01: ❗️Failed 
  Email     : Hi Sarah, we'd love to speak with you about the Software Eng...
  category  -> Got: interview_invitation     Expected: interview_invitation      ✅
  urgency   -> Got: medium                   Expected: high                      ⚠️
  job_field -> Got: software_engineering     Expected: software_engineering      ✅
  matched   : 2/3

Test 02: ✅PASS 
  Email     : Thank you again for your time. After reviewing everything, w...
  category  -> Got: rejection                Expected: rejection                 ✅
  urgency   -> Got: low                      Expected: low                       ✅
  job_field -> Got: general                  Expected: general                   ✅
  matched   : 3/3

Test 03: ✅PASS 
  Email     : I found your background while searching for candidates and t...
  category  -> Got: recruiter_outreach       Expected: recruiter_outreach        ✅
  urgency   -> Got: low                      Expected: low              

---
# Save best models as .pkl file
---

In [29]:
models_dir = BASE_DIR / "models"
models_dir.mkdir(exist_ok=True)

joblib.dump(best_category_model, models_dir / "category_model.pkl")
joblib.dump(best_urgency_model, models_dir / "urgency_model.pkl")
joblib.dump(best_job_field_model, models_dir / "job_field_model.pkl")

print(f"category_model.pkl: {best_category_name}")
print(f"urgency_model.pkl: {best_urgency_name}")
print(f"job_field_model.pkl: {best_job_field_name}")
print(f"\nSaved to: {models_dir}")

category_model.pkl: LogisticRegression
urgency_model.pkl: LogisticRegression
job_field_model.pkl: LinearSVC

Saved to: /Users/ilkerbaran/PycharmProjects/interview_intel/ml/models
